# Modelos

In [44]:
import pickle
from abc import abstractmethod
from pathlib import Path
from typing import Union

class BaseModel:
    """Base class for all models with serialization capabilities."""

    @abstractmethod
    def predict(self, X):
        pass

    def serialize(self, file_path: Union[str, Path]) -> None:
        """
        Serialize the model to a file using pickle.

        Args:
            file_path: Path where the model will be saved.
        """
        print(f"Saving model in {file_path}")
        with open(file_path, 'wb') as f:
            pickle.dump(self, f)

    @classmethod
    def deserialize(cls, file_path: Union[str, Path]) -> 'BaseModel':
        """
        Deserialize a model from a file.

        Args:
            file_path: Path to the serialized model.

        Returns:
            The deserialized model instance.
        """
        with open(file_path, 'rb') as f:
            return pickle.load(f)

In [45]:
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix


class AdaBoost(BaseModel):
    """
    A wrapper class for scikit-learn's AdaBoost algorithms
    for both regression and classification tasks.
    AdaBoost (Adaptive Boosting) is a boosting meta-estimator that begins by fitting
    a model on the original dataset and then fits additional copies of the model
    on the same dataset but where the weights of incorrectly classified data points
    are adjusted such that subsequent models focus more on difficult cases.
    Inherits from BaseModel.
    """
    def __init__(self, problem_type='classification', base_estimator=None, **kwargs):
        """
        Initializes the AdaBoost class.

        Args:
            problem_type (str, optional): The type of problem ('classification' or 'regression').
                                         Defaults to 'classification'.
            base_estimator (estimator, optional): The base estimator from which the boosted
                                                  ensemble is built. If None, the default base
                                                  estimator is DecisionTreeClassifier(max_depth=1)
                                                  for classification and DecisionTreeRegressor(max_depth=3)
                                                  for regression. Defaults to None.
            **kwargs: Additional arguments to be passed to the scikit-learn
                      AdaBoost Classifier or Regressor. Common arguments include:
                      n_estimators (number of boosting stages), learning_rate, random_state, etc.
        """
        self.problem_type = problem_type.lower()
        self.base_estimator = base_estimator # Store the base estimator if provided
        self.model = self._create_model(**kwargs)

    def _create_model(self, **kwargs):
        """
        Creates the appropriate AdaBoost model instance
        based on the problem type and optional base estimator.
        """
        if self.problem_type == 'classification':
            # Default base estimator for AdaBoostClassifier is DecisionTreeClassifier(max_depth=1)
            # We pass the provided base_estimator or let scikit-learn use its default if None
            return AdaBoostClassifier(estimator=self.base_estimator, **kwargs)
        elif self.problem_type == 'regression':
            # Default base estimator for AdaBoostRegressor is DecisionTreeRegressor(max_depth=3)
            # We pass the provided base_estimator or let scikit-learn use its default if None
            return AdaBoostRegressor(estimator=self.base_estimator, **kwargs)
        else:
            raise ValueError("problem_type must be 'classification' or 'regression'")

    def fit(self, X, y):
        """
        Trains the AdaBoost model with the provided training data.

        Args:
            X (array-like): Training data (features).
            y (array-like): Training labels or target values.
        """
        self.model.fit(X, y)

    def predict(self, X):
        """
        Performs predictions on the given data.

        Args:
            X (array-like): Data for prediction (features).

        Returns:
            array-like: Model predictions.
        """
        return self.model.predict(X)

    def predict_proba(self, X):
        """
        Predicts class probabilities for classification tasks.
        Note: predict_proba for AdaBoostClassifier can be less reliable than for
        other models like RandomForest, as it's based on combining confidence scores
        of weak learners.

        Args:
            X (array-like): Data for prediction (features).

        Returns:
            array-like: Predicted class probabilities.

        Raises:
            NotImplementedError: If the problem type is not 'classification'.
        """
        if self.problem_type != 'classification':
            raise NotImplementedError("predict_proba is only available for classification tasks.")
        # Check if the underlying model supports predict_proba (AdaBoostClassifier does)
        if hasattr(self.model, 'predict_proba'):
             return self.model.predict_proba(X)
        else:
             raise NotImplementedError("The wrapped AdaBoost model does not support predict_proba.")


    def evaluate(self, X_test, y_test):
        """
        Evaluates the performance of the model on the provided test data.

        Args:
            X_test (array-like): Test data (features).
            y_test (array-like): Test labels or target values.

        Returns:
            dict: A dictionary containing the appropriate evaluation metrics
                  for the problem type.
        """
        y_pred = self.predict(X_test)
        results = {}
        if self.problem_type == 'regression':
            results['mean_squared_error'] = mean_squared_error(y_test, y_pred)
            results['r2_score'] = r2_score(y_test, y_pred)
            # You could add more regression metrics here if needed, e.g., MAE, RMSE
            # from sklearn.metrics import mean_absolute_error
            # results['mean_absolute_error'] = mean_absolute_error(y_test, y_pred)
            # results['root_mean_squared_error'] = mean_squared_error(y_test, y_pred, squared=False) # For RMSE
        elif self.problem_type == 'classification':
            results['accuracy'] = accuracy_score(y_test, y_pred)
            # Use output_dict=True for easier parsing of classification report
            results['classification_report'] = classification_report(y_test, y_pred, output_dict=True)
            # Convert confusion matrix to list for potentially easier handling if serializing
            results['confusion_matrix'] = confusion_matrix(y_test, y_pred).tolist()
            # You could add more classification metrics here if needed
            # from sklearn.metrics import precision_score, recall_score, f1_score
            # results['precision_macro'] = precision_score(y_test, y_pred, average='macro')
            # results['recall_macro'] = recall_score(y_test, y_pred, average='macro')
            # results['f1_macro'] = f1_score(y_test, y_pred, average='macro')
        return results

    def get_feature_importance(self):
        """
        Returns the feature importances.
        Note: Feature importances for AdaBoost are calculated based on the
        contributions of the base estimators.

        Returns:
            numpy.ndarray: An array containing the importance of each feature.
                           Returns None if the model does not support feature importance.
        """
        # AdaBoost models have feature_importances_ attribute if the base estimator does
        # and the model is a classifier or regressor.
        if hasattr(self.model, 'feature_importances_'):
            return self.model.feature_importances_
        else:
             # This might happen if the base estimator doesn't support feature_importances_
             return None

In [46]:
import torch.nn as nn

class BaseNeuralNetwork(nn.Module, BaseModel):
    def __init__(self, input_dim, output_dim, task_type='regression'):
        super().__init__()
        self.task_type = task_type

        # Define 10-layer fully connected neural network with ReLU activations between layers
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),  # Layer 1 - wider to capture more patterns
            nn.ReLU(),
            nn.Dropout(0.3),  # Add dropout for regularization
            nn.BatchNorm1d(256),  # Add batch normalization for better training

            nn.Linear(256, 128),  # Layer 2
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.BatchNorm1d(128),

            nn.Linear(128, output_dim)
        )

        # For classification, use softmax on output
        if task_type == 'classification':
            self.activation = nn.Softmax(dim=1)
        else:
            self.activation = None

    def forward(self, x):
        out = self.model(x)
        if self.activation:
            out = self.activation(out)
        return out

In [47]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix


class DecisionTree(BaseModel):
    """
    A wrapper class for scikit-learn's Decision Tree algorithms
    for both regression and classification tasks.
    """
    def __init__(self, problem_type='classification', **kwargs):
        """
        Initializes the CustomDecisionTree class.

        Args:
            problem_type (str, optional): The type of problem ('classification' or 'regression').
                                         Defaults to 'classification'.
            **kwargs: Additional arguments to be passed to the scikit-learn
                      Decision Tree Classifier or Regressor.
        """
        self.problem_type = problem_type.lower()
        self.model = self._create_model(**kwargs)

    def _create_model(self, **kwargs):
        """
        Creates the appropriate Decision Tree model instance
        based on the problem type.
        """
        if self.problem_type == 'classification':
            return DecisionTreeClassifier(**kwargs)
        elif self.problem_type == 'regression':
            return DecisionTreeRegressor(**kwargs)
        else:
            raise ValueError("problem_type must be 'classification' or 'regression'")

    def fit(self, X, y):
        """
        Trains the Decision Tree model with he provided training data.

        Args:
            X (array-like): Training data (features).
            y (array-like): Training labels or target values.
        """
        self.model.fit(X, y)

    def predict(self, X):
        """
        Performs predictions on the given data.

        Args:
            X (array-like): Data for prediction (features).

        Returns:
            array-like: Model predictions.
        """
        return self.model.predict(X)


    def predict_proba(self, X):
        if self.problem_type != 'classification':
            raise NotImplementedError("predict_proba is only available for classification tasks.")
        return self.model.predict_proba(X)


    def evaluate(self, X_test, y_test):
        """
        Evaluates the performance of the model on the provided test data.

        Args:
            X_test (array-like): Test data (features).
            y_test (array-like): Test labels or target values.

        Returns:
            dict: A dictionary containing the appropriate evaluation metrics
                  for the problem type.
        """
        y_pred = self.predict(X_test)
        results = {}
        if self.problem_type == 'regression':
            results['mean_squared_error'] = mean_squared_error(y_test, y_pred)
            results['r2_score'] = r2_score(y_test, y_pred)
        elif self.problem_type == 'classification':
            results['accuracy'] = accuracy_score(y_test, y_pred)
            results['classification_report'] = classification_report(y_test, y_pred)
            results['confusion_matrix'] = confusion_matrix(y_test, y_pred)
        return results

    def get_feature_importance(self):
        """
        Returns the feature importances (only for tree-based models).

        Returns:
            numpy.ndarray: An array containing the importance of each feature.
                           Returns None if the model does not support feature importance.
        """
        if hasattr(self.model, 'feature_importances_'):
            return self.model.feature_importances_
        else:
            return None

In [48]:
import pandas as pd
import numpy as np
from typing import Union, Any, Dict
import matplotlib.pyplot as plt
import seaborn as sns
import hdbscan
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline



class HDBSCANMixed(BaseModel):  # Renomeando a classe para refletir o algoritmo
    """
    A density-based clustering algorithm using HDBSCAN* which can handle mixed
    numerical and categorical data via intelligent pre-processing.

    HDBSCAN* is a robust and efficient algorithm that extends DBSCAN by not
    requiring the 'eps' parameter and being able to find clusters of varying densities.
    It builds a hierarchy of clusters and extracts the most stable ones.
    """

    def __init__(self, min_cluster_size: int = 5, min_samples: Union[int, None] = None,
                 cluster_selection_epsilon: float = 0.0, metric: str = 'euclidean', **hdbscan_kwargs):
        """
        Initializes the HDBSCANMixed clustering model.

        Args:
            min_cluster_size (int): The minimum size of clusters. Smaller values will allow
                more and smaller clusters to be formed, and can be more sensitive to noise.
                Defaults to 5.
            min_samples (int, optional): The number of samples in a neighborhood for a point to
                be considered a core point. This controls the "conservativeness" of the clustering.
                Defaults to None (uses min_cluster_size).
            cluster_selection_epsilon (float): A threshold for cluster merging. Clusters below
                this threshold in the hierarchy will be merged. Defaults to 0.0.
            metric (str): The metric to use when calculating the distance between data points.
                Since HDBSCAN will be applied after numerical transformation (One-Hot Encoding
                + Scaling), 'euclidean' is typically the default. Other metrics like 'cityblock',
                'cosine', 'minkowski' (with p=1 or 2) can also be used. Defaults to 'euclidean'.
            **hdbscan_kwargs: Additional keyword arguments to pass directly to the hdbscan.HDBSCAN constructor.
                Useful for fine-tuning, e.g., 'prediction_data=True' if you plan to predict on new data.
        """
        if not isinstance(min_cluster_size, int) or min_cluster_size <= 0:
            raise ValueError("min_cluster_size must be a positive integer.")
        if min_samples is not None and (not isinstance(min_samples, int) or min_samples <= 0):
            raise ValueError("min_samples must be a positive integer or None.")
        if not isinstance(cluster_selection_epsilon, (int, float)) or cluster_selection_epsilon < 0:
            raise ValueError("cluster_selection_epsilon must be a non-negative float.")

        self.min_cluster_size = min_cluster_size
        self.min_samples = min_samples
        self.cluster_selection_epsilon = cluster_selection_epsilon
        self.metric = metric
        self.hdbscan_kwargs = hdbscan_kwargs

        self.labels = None
        self._fitted_pipeline = None  # Store the fitted sklearn pipeline
        self.feature_names_original = None

    def _build_preprocessing_pipeline(self, X: pd.DataFrame) -> Pipeline:
        """
        Builds the scikit-learn pipeline for pre-processing mixed data.
        """
        numeric_features = X.select_dtypes(include=np.number).columns.tolist()
        categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

        # Create a preprocessor using ColumnTransformer
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', StandardScaler(), numeric_features),
                ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
            ],
            remainder='passthrough'  # Keep other columns if any, though usually not needed
        )

        # Create the HDBSCAN model
        hdbscan_model = hdbscan.HDBSCAN(
            min_cluster_size=self.min_cluster_size,
            min_samples=self.min_samples,
            cluster_selection_epsilon=self.cluster_selection_epsilon,
            metric=self.metric,
            **self.hdbscan_kwargs
        )

        # Build the full pipeline
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('hdbscan', hdbscan_model)
        ])
        return pipeline

    def fit(self, X: pd.DataFrame) -> None:
        """
        Performs HDBSCAN clustering on the input data after appropriate pre-processing.

        Args:
            X (pd.DataFrame): The input DataFrame containing the data to cluster.
                It should contain both numerical and categorical features.

        Returns:
            None: The cluster labels are stored in the `self.labels` attribute.
        """
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Input X must be a pandas DataFrame.")

        self.feature_names_original = X.columns.tolist()
        self._fitted_pipeline = self._build_preprocessing_pipeline(X)

        print("Iniciando o pré-processamento e treinamento HDBSCAN...")
        self._fitted_pipeline.fit(X)
        print("Treinamento concluído.")

        self.labels = self._fitted_pipeline.named_steps['hdbscan'].labels_

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        """
        Predicts cluster labels for new, unseen data points using the fitted HDBSCAN model.
        Note: HDBSCAN's prediction is different from K-Means. It typically assigns new points
        to the cluster of their closest training point if within a certain density, or to noise.
        This requires `prediction_data=True` in the HDBSCAN constructor during fit.

        Args:
            X (pd.DataFrame): The new data points to predict cluster labels for.

        Returns:
            np.ndarray: An array of cluster labels for each new data point.
                -1 indicates noise.
        """
        if self._fitted_pipeline is None:
            raise ValueError("Model not fitted yet. Call 'fit' first.")

        # Ensure that prediction_data=True was set during initialization if you want a robust predict
        if 'prediction_data' not in self.hdbscan_kwargs or not self.hdbscan_kwargs['prediction_data']:
            print("Warning: For robust prediction with HDBSCAN, initialize with prediction_data=True.")
            print("Falling back to a simpler prediction method: transforming data and then calling predict.")
            # For simpler prediction, just transform new data and use the hdbscan predict method
            # This method usually finds the closest existing cluster point and assigns.
            X_transformed = self._fitted_pipeline.named_steps['preprocessor'].transform(X)
            return self._fitted_pipeline.named_steps['hdbscan'].predict(X_transformed)

        # If prediction_data was true, the hdbscan model is ready for robust prediction
        X_transformed = self._fitted_pipeline.named_steps['preprocessor'].transform(X)
        return self._fitted_pipeline.named_steps['hdbscan'].predict(X_transformed)

    def evaluate(self, X_test: Union[pd.DataFrame, None] = None,
                 y_test: Union[np.ndarray, pd.Series, None] = None) -> Dict[str, Any]:
        """
        Evaluates the clustering performance using external evaluation metrics
        if ground truth labels are provided.

        Args:
            X_test (pd.DataFrame, optional): Test data (not used for evaluation
                in unsupervised clustering, included for API consistency).
                Defaults to None.
            y_test (np.ndarray, pd.Series, optional): Ground truth cluster labels
                for the data used in the fit method. Defaults to None.

        Returns:
            dict: A dictionary containing evaluation metrics. Currently includes
            Adjusted Rand Score and Normalized Mutual Information if y_test
            is provided and the number of labels matches.
        """
        results = {}
        if y_test is not None and self.labels is not None and len(self.labels) == len(y_test):
            if isinstance(y_test, pd.Series):
                y_test = y_test.values

            # Filter out noise points (-1) from both predicted and true labels for evaluation
            non_noise_indices = np.where(self.labels != -1)[0]
            if len(non_noise_indices) > 0:
                filtered_labels = self.labels[non_noise_indices]
                filtered_y_test = y_test[non_noise_indices]

                if len(np.unique(filtered_labels)) > 1 and len(np.unique(filtered_y_test)) > 1:
                    results['adjusted_rand_score'] = adjusted_rand_score(filtered_y_test, filtered_labels)
                    results['normalized_mutual_info_score'] = normalized_mutual_info_score(filtered_y_test,
                                                                                           filtered_labels)
                else:
                    print("Warning: Not enough unique labels after filtering noise for ARI/NMI calculation.")
            else:
                print("Warning: All points labeled as noise, cannot calculate ARI/NMI.")
        return results

    def get_centroids(self) -> None:
        """
        Returns the centroids of the clusters.

        Density-based methods like HDBSCAN* do not typically have well-defined
        centroids in the same way as partition-based or model-based methods.
        Returns None.
        """
        return None

    def plot_clusters(self, X: pd.DataFrame, title: str = "HDBSCAN Clustering") -> None:
        """
        Visualizes the clusters formed by HDBSCAN*.

        Args:
            X (pd.DataFrame): The data used for clustering (features).
            title (str): Title of the plot.
        """
        if self.labels is None:
            raise RuntimeError("Model has not been fitted yet. Call fit() before plotting.")

        X_df = X.copy()
        X_df['cluster'] = self.labels

        # Identify numerical and categorical features
        numerical_features = [col for col in X_df.columns if
                              pd.api.types.is_numeric_dtype(X_df[col]) and col != 'cluster']
        categorical_features = [col for col in X_df.columns if
                                not pd.api.types.is_numeric_dtype(X_df[col]) and col != 'cluster']

        # --- Numerical Features Visualization (Scatter Plots) ---
        if len(numerical_features) >= 2:
            print(f"\n--- Visualizing Numerical Features for {title} ---")

            X_df['cluster_str'] = X_df['cluster'].astype(str)
            X_df['cluster_str'] = X_df['cluster_str'].replace('-1', 'Noise')

            unique_clusters = sorted(X_df['cluster_str'].unique())
            if 'Noise' in unique_clusters:
                unique_clusters.remove('Noise')
                unique_clusters.append('Noise')

            palette = sns.color_palette("viridis",
                                        n_colors=len(unique_clusters) - (1 if 'Noise' in unique_clusters else 0))
            if 'Noise' in unique_clusters:
                palette.append('gray')
            cluster_palette = dict(zip(unique_clusters, palette))

            if len(numerical_features) >= 2:
                plt.figure(figsize=(10, 8))
                sns.scatterplot(
                    x=numerical_features[0],
                    y=numerical_features[1],
                    hue='cluster_str',
                    style='cluster_str',
                    data=X_df,
                    palette=cluster_palette,
                    s=80,
                    alpha=0.7
                )
                plt.title(f'Cluster Visualization of {numerical_features[0]} vs {numerical_features[1]}\n{title}')
                plt.xlabel(numerical_features[0])
                plt.ylabel(numerical_features[1])
                plt.legend(title='Cluster')
                plt.grid(True, linestyle='--', alpha=0.6)
                plt.show()
            else:
                print("Not enough numerical features (at least 2 required) for scatter plot visualization.")

        # --- Categorical Features Visualization (Count Plots) ---
        if categorical_features:
            print(f"\n--- Visualizing Categorical Features for {title} ---")
            num_categorical_plots = len(categorical_features)
            cols = 2
            rows = (num_categorical_plots + cols - 1) // cols
            fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
            axes = axes.flatten()

            for i, feature in enumerate(categorical_features):
                sns.countplot(x=feature, hue='cluster_str', data=X_df, ax=axes[i], palette=cluster_palette)
                axes[i].set_title(f'{feature} Distribution by Cluster')
                axes[i].set_xlabel(feature)
                axes[i].set_ylabel('Count')
                axes[i].tick_params(axis='x', rotation=45)
                axes[i].legend(title='Cluster')

            for j in range(i + 1, len(axes)):
                fig.delaxes(axes[j])

            plt.tight_layout()
            plt.suptitle(f"Distribution of Categorical Features Across Clusters\n{title}", y=1.02, fontsize=16)
            plt.show()
        else:
            print("No categorical features to visualize.")

        print("\n--- Cluster Sizes ---")
        cluster_counts = X_df['cluster_str'].value_counts().sort_index()
        print(cluster_counts)

In [49]:
import numpy as np
import pandas as pd
from typing import Any, Optional, Tuple
from numba import njit, prange
from numba.typed import List as NumbaList
from collections import deque


class KDTreeNode:
    __slots__ = ('point', 'label', 'split_dim', 'left', 'right', 'idx')

    def __init__(self, point: np.ndarray, label: Any, split_dim: int,
                 left: Optional['KDTreeNode'], right: Optional['KDTreeNode'], idx: int):
        self.point = point
        self.label = label
        self.split_dim = split_dim
        self.left = left
        self.right = right
        self.idx = idx


@njit(cache=True, fastmath=True)
def _knn_search_core(points_arr: np.ndarray, labels_arr: np.ndarray, split_dims_arr: np.ndarray,
                     k: int, query_point: np.ndarray, n_nodes: int,
                     tree_depth_unused: int) -> np.ndarray:
    best_dist2s = np.full(k, np.inf, dtype=np.float64)
    best_labels = np.full(k, -1.0, dtype=np.float64)
    node_stack = NumbaList()
    if n_nodes > 0 and split_dims_arr[0] != -1:
        node_stack.append(0)
    while len(node_stack) > 0:
        current_node_idx = node_stack.pop()
        if split_dims_arr[current_node_idx] == -1:
            continue
        point = points_arr[current_node_idx]
        label = labels_arr[current_node_idx]
        split_dim = split_dims_arr[current_node_idx]
        diff = query_point - point
        d2 = diff.dot(diff)
        idx_max_dist = 0
        for i in range(1, k):
            if best_dist2s[i] > best_dist2s[idx_max_dist]:
                idx_max_dist = i
        current_max_k_dist2 = best_dist2s[idx_max_dist]
        if d2 < current_max_k_dist2:
            best_dist2s[idx_max_dist] = d2
            best_labels[idx_max_dist] = label
            new_idx_max_dist = 0
            for i in range(1, k):
                if best_dist2s[i] > best_dist2s[new_idx_max_dist]:
                    new_idx_max_dist = i
            current_max_k_dist2 = best_dist2s[new_idx_max_dist]
        query_val_at_dim = query_point[split_dim]
        point_val_at_dim = point[split_dim]
        left_child_idx = 2 * current_node_idx + 1
        right_child_idx = 2 * current_node_idx + 2
        if query_val_at_dim <= point_val_at_dim:
            near_child_idx, far_child_idx = left_child_idx, right_child_idx
        else:
            near_child_idx, far_child_idx = right_child_idx, left_child_idx
        if near_child_idx < n_nodes and split_dims_arr[near_child_idx] != -1:
            node_stack.append(near_child_idx)
        dist_to_plane_sq = (query_val_at_dim - point_val_at_dim) ** 2
        if far_child_idx < n_nodes and split_dims_arr[far_child_idx] != -1 and \
                dist_to_plane_sq < current_max_k_dist2:
            node_stack.append(far_child_idx)
    return best_labels


def _build_kdtree_recursive_nodes(
        current_indices: np.ndarray,
        depth: int,
        X_data: np.ndarray,
        y_data_numeric_float64: np.ndarray,
        dataset_dim: int
) -> Tuple[Optional[KDTreeNode], int]:
    """
    Recursive helper to build KDTreeNode structure.
    Returns the node and the max depth reached in this branch.
    """
    current_branch_max_depth = depth
    n = len(current_indices)
    if n == 0:
        return None, current_branch_max_depth

    axis = depth % dataset_dim

    current_X_subset = X_data[current_indices]  # Data for points in current_indices

    mid_idx_in_subset = n // 2
    # `partitioned_indices_in_subset` are indices relative to `current_X_subset` (and thus `current_indices`)
    partitioned_indices_in_subset = np.argpartition(
        current_X_subset[:, axis], mid_idx_in_subset, kind='introselect'
    )

    # Index of median within `current_indices` array
    median_idx_in_current_indices_array = partitioned_indices_in_subset[mid_idx_in_subset]
    # Actual index in the original full X_data, y_data_numeric_float64
    median_original_overall_idx = current_indices[median_idx_in_current_indices_array]

    # Get subsets of `current_indices` for left and right children
    left_original_indices = current_indices[partitioned_indices_in_subset[:mid_idx_in_subset]]
    right_original_indices = current_indices[partitioned_indices_in_subset[mid_idx_in_subset + 1:]]

    left_node, left_max_depth = _build_kdtree_recursive_nodes(
        left_original_indices, depth + 1, X_data, y_data_numeric_float64, dataset_dim
    )
    current_branch_max_depth = max(current_branch_max_depth, left_max_depth)

    right_node, right_max_depth = _build_kdtree_recursive_nodes(
        right_original_indices, depth + 1, X_data, y_data_numeric_float64, dataset_dim
    )
    current_branch_max_depth = max(current_branch_max_depth, right_max_depth)

    node = KDTreeNode(
        point=X_data[median_original_overall_idx],
        label=y_data_numeric_float64[median_original_overall_idx],
        split_dim=axis,
        left=left_node,
        right=right_node,
        idx=-1  # Placeholder, will be set during BFS flattening
    )
    return node, current_branch_max_depth


def build_kdtree_iterative_flat(X: np.ndarray, y_numeric_float64: np.ndarray, verbose: bool = False) -> Tuple[
    np.ndarray, np.ndarray, np.ndarray, int, int]:
    n_samples, dim = X.shape
    if n_samples == 0:
        return (np.empty((0, dim), dtype=np.float64),
                np.empty(0, dtype=np.float64),
                np.empty(0, dtype=np.int32),
                0, 0)

    if verbose: print("Building KD-Tree node structure...")
    # Call the module-level recursive helper
    root_node, max_tree_depth = _build_kdtree_recursive_nodes(
        np.arange(n_samples), 0, X, y_numeric_float64, dim
    )

    if root_node is None:  # Should only happen if n_samples was 0, already handled.
        return (np.empty((0, dim), dtype=np.float64),
                np.empty(0, dtype=np.float64),
                np.empty(0, dtype=np.int32),
                0, 0)  # max_tree_depth would be 0

    if verbose: print(f"KDTreeNode structure built. Max depth: {max_tree_depth}. Starting flattening...")

    # Flatten tree via BFS into numpy arrays
    flat_idx_to_kdtreenode = {}
    bfs_queue = deque()

    if root_node:
        bfs_queue.append((root_node, 0))
        flat_idx_to_kdtreenode[0] = root_node

    max_flat_idx_reached = 0

    while bfs_queue:
        current_kdtree_node, current_flat_idx = bfs_queue.popleft()
        max_flat_idx_reached = max(max_flat_idx_reached, current_flat_idx)

        if current_kdtree_node.left:
            left_flat_idx = 2 * current_flat_idx + 1
            flat_idx_to_kdtreenode[left_flat_idx] = current_kdtree_node.left
            bfs_queue.append((current_kdtree_node.left, left_flat_idx))
        if current_kdtree_node.right:
            right_flat_idx = 2 * current_flat_idx + 2
            flat_idx_to_kdtreenode[right_flat_idx] = current_kdtree_node.right
            bfs_queue.append((current_kdtree_node.right, right_flat_idx))

    num_flat_array_elements = max_flat_idx_reached + 1
    actual_num_nodes = len(flat_idx_to_kdtreenode)

    if verbose:
        print(f"Flattening complete: {actual_num_nodes} actual nodes. Flat array size: {num_flat_array_elements}.")

    points_arr = np.full((num_flat_array_elements, dim), np.nan, dtype=np.float64)
    labels_arr = np.full(num_flat_array_elements, np.nan, dtype=np.float64)
    split_dims_arr = np.full(num_flat_array_elements, -1, dtype=np.int32)

    for flat_idx, node_obj in flat_idx_to_kdtreenode.items():
        points_arr[flat_idx] = node_obj.point
        labels_arr[flat_idx] = node_obj.label
        split_dims_arr[flat_idx] = node_obj.split_dim

    return points_arr, labels_arr, split_dims_arr, num_flat_array_elements, max_tree_depth


class KNN(BaseModel):
    def __init__(self, k: int = 5, task: str = 'classification', verbose: bool = False):
        super().__init__()
        if task not in ('classification', 'regression'):
            raise ValueError("Task must be 'classification' or 'regression'.")
        if k < 1:
            raise ValueError("k must be at least 1.")
        self.k = k
        self.task = task
        self.verbose = verbose

        self.tree_points: Optional[np.ndarray] = None
        self.tree_labels: Optional[np.ndarray] = None
        self.tree_split_dims: Optional[np.ndarray] = None
        self.n_tree_nodes_allocated: int = 0
        self.tree_depth: int = 0

        self.categories_: Optional[pd.Index] = None
        self.label_mapper_: Optional[pd.Index] = None

    def fit(self, X: np.ndarray, y: Any) -> None:
        X_arr = np.ascontiguousarray(X, dtype=np.float64)

        y_np: np.ndarray
        self.categories_ = None
        self.label_mapper_ = None

        if isinstance(y, pd.Series) and isinstance(y.dtype, pd.CategoricalDtype):
            if self.verbose: print("Input y is pandas Series with CategoricalDtype.")
            self.categories_ = y.cat.categories
            y_np = y.to_numpy()
        elif hasattr(y, 'to_numpy'):
            if self.verbose: print("Input y is pandas Series or similar.")
            y_np = y.to_numpy()
        else:
            if self.verbose: print("Input y is NumPy array or list.")
            y_np = np.asarray(y)

        y_arr: np.ndarray
        if self.task == 'regression':
            if not np.issubdtype(y_np.dtype, np.number):
                raise ValueError(f"Regression task requires numeric labels. Got dtype {y_np.dtype}.")
            y_arr = y_np.astype(np.float64)
        else:
            if np.issubdtype(y_np.dtype, np.number):
                if self.verbose: print(f"Numeric labels detected for classification (dtype: {y_np.dtype}).")
                y_arr = y_np.astype(np.float64)
            else:
                if self.verbose: print(f"Non-numeric labels (dtype: {y_np.dtype}) for classification. Factorizing.")
                integer_codes, uniques = pd.factorize(y_np, sort=True)
                self.label_mapper_ = uniques
                y_arr = integer_codes.astype(np.float64)

        y_arr = np.ascontiguousarray(y_arr)

        if X_arr.shape[0] == 0:
            raise ValueError("Cannot fit on empty data X.")
        if X_arr.shape[0] != y_arr.shape[0]:
            raise ValueError(
                f"X and y must have the same number of samples. X has {X_arr.shape[0]}, y processed to {y_arr.shape[0]}.")

        if self.k > X_arr.shape[0]:
            if self.verbose:
                print(
                    f"Warning: k ({self.k}) is greater than number of samples ({X_arr.shape[0]}). Setting k to {X_arr.shape[0]}.")
            self.k = X_arr.shape[0]

        if X_arr.shape[0] > 0 and self.k == 0:
            self.k = 1

        self.tree_points, self.tree_labels, self.tree_split_dims, \
            self.n_tree_nodes_allocated, self.tree_depth = \
            build_kdtree_iterative_flat(X_arr, y_arr, self.verbose)

        if self.verbose:
            actual_nodes_count = np.sum(
                self.tree_split_dims != -1) if self.tree_split_dims is not None and self.tree_split_dims.size > 0 else 0
            print(
                f"KD-Tree built. Flat array size: {self.n_tree_nodes_allocated}. Actual nodes: {actual_nodes_count}. Dimensions={X_arr.shape[1]}. Max depth: {self.tree_depth}.")

    def predict(self, X: np.ndarray) -> np.ndarray:
        if self.tree_points is None or self.n_tree_nodes_allocated == 0:
            n_queries = X.shape[0]
            if self.task == 'regression':
                return np.full(n_queries, np.nan, dtype=np.float64)
            else:
                return np.full(n_queries, None, dtype=object)

        X_arr = np.ascontiguousarray(X, dtype=np.float64)
        n_queries = X_arr.shape[0]

        k_for_predict = max(1, self.k) if self.n_tree_nodes_allocated > 0 else self.k
        if k_for_predict == 0:
            if self.task == 'regression':
                return np.full(n_queries, np.nan, dtype=np.float64)
            else:
                return np.full(n_queries, None, dtype=object)

        preds_numeric = np.empty(n_queries, dtype=np.float64)

        for i in prange(n_queries):
            query_point = X_arr[i]
            neighbor_labels = _knn_search_core(
                self.tree_points, self.tree_labels, self.tree_split_dims,
                k_for_predict, query_point, self.n_tree_nodes_allocated, self.tree_depth
            )
            valid_neighbor_labels = neighbor_labels[neighbor_labels != -1.0]
            if np.isnan(valid_neighbor_labels).any():
                valid_neighbor_labels = valid_neighbor_labels[~np.isnan(valid_neighbor_labels)]

            if len(valid_neighbor_labels) == 0:
                preds_numeric[i] = np.nan
                continue

            if self.task == 'classification':
                u_labels, u_counts = np.unique(valid_neighbor_labels, return_counts=True)
                preds_numeric[i] = u_labels[np.argmax(u_counts)]
            else:
                preds_numeric[i] = np.mean(valid_neighbor_labels)

        if self.verbose and n_queries > 0:
            print(f"Numeric predictions generated. Processed {n_queries}/{n_queries}.")

        if self.task == 'classification':
            output_dtype = object
            if self.label_mapper_ is not None:
                output_dtype = self.label_mapper_.dtype
            elif self.categories_ is not None:
                output_dtype = self.categories_.dtype

            final_preds = np.empty(n_queries, dtype=output_dtype)

            for i in range(n_queries):
                numeric_pred_val = preds_numeric[i]
                if np.isnan(numeric_pred_val):
                    final_preds[i] = np.nan if pd.api.types.is_numeric_dtype(final_preds.dtype) else None
                    continue

                if self.label_mapper_ is not None:
                    int_code = int(round(numeric_pred_val))
                    if 0 <= int_code < len(self.label_mapper_):
                        final_preds[i] = self.label_mapper_[int_code]
                    else:
                        final_preds[i] = np.nan if pd.api.types.is_numeric_dtype(final_preds.dtype) else None
                elif self.categories_ is not None:
                    if pd.api.types.is_integer_dtype(self.categories_.dtype):
                        final_preds[i] = int(round(numeric_pred_val))
                    elif pd.api.types.is_float_dtype(self.categories_.dtype):
                        final_preds[i] = float(numeric_pred_val)
                    elif pd.api.types.is_bool_dtype(self.categories_.dtype):
                        final_preds[i] = bool(round(numeric_pred_val))
                    else:
                        final_preds[i] = numeric_pred_val
                else:
                    if numeric_pred_val == round(numeric_pred_val):
                        final_preds[i] = int(round(numeric_pred_val))
                    else:
                        final_preds[i] = float(numeric_pred_val)
            if self.verbose and n_queries > 0: print("Classification labels mapped back to original types.")
            return final_preds
        else:
            if self.verbose and n_queries > 0: print("Regression predictions complete.")
            return preds_numeric

In [50]:
import pandas as pd
import numpy as np
from kmodes.kprototypes import KPrototypes
from typing import List, Union, Any, Tuple, Dict
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import matplotlib.pyplot as plt
import seaborn as sns


class KPrototypesModel(BaseModel):
    """
    A wrapper class for the kmodes KPrototypes algorithm for clustering
    mixed numerical and categorical data.
    """
    def __init__(self, n_clusters: int, categorical_indices: List[int], **kwargs: Any):
        """
        Initializes the KPrototypesModel.

        Args:
            n_clusters: The number of clusters to form.
            categorical_indices: Indices of categorical columns in the input data.
            **kwargs: Additional arguments for the KPrototypes constructor.
        """
        if not isinstance(n_clusters, int) or n_clusters <= 0:
            raise ValueError("n_clusters must be a positive integer.")
        if not isinstance(categorical_indices, list) or not all(isinstance(i, int) for i in categorical_indices):
            raise ValueError("categorical_indices must be a list of integers.")

        self.n_clusters = n_clusters
        self.categorical_indices = sorted(categorical_indices)
        self.model = KPrototypes(n_clusters=n_clusters, **kwargs)
        self.problem_type = 'clustering'
        self.feature_names = None # To store feature names for better plots


    def fit(self, X: Union[pd.DataFrame, np.ndarray], y: Any = None) -> None:
        """
        Trains the KPrototypes model.

        Args:
            X: Training data (features).
            y: Ignored (unsupervised learning).
        """
        if isinstance(X, pd.DataFrame):
            self.feature_names = X.columns.tolist()
        else:
            self.feature_names = [f"feature_{i}" for i in range(X.shape[1])]

        X_array = self._validate_data(X)
        if X_array.shape[1] <= max(self.categorical_indices) if self.categorical_indices else False:
            raise ValueError("Number of columns in X must be greater than the maximum index in categorical_indices.")
        self.model.fit(X_array, categorical=self.categorical_indices)

    def predict(self, X: Union[pd.DataFrame, np.ndarray]) -> np.ndarray:
        """
        Predicts cluster assignments for new data.

        Args:
            X: New data to predict clusters for.

        Returns:
            Array of cluster labels for each data point.
        """
        X_array = self._validate_data(X)
        # Check if the model has been fitted and if the number of features matches
        if hasattr(self.model, 'cluster_centroids_'):
            num_numerical_features = self.model.cluster_centroids_[0].shape[1]
            num_categorical_features = self.model.cluster_centroids_[1].shape[1]
            # The total number of features used for training is the sum of numerical and categorical
            total_trained_features = num_numerical_features + num_categorical_features
            if X_array.shape[1] != total_trained_features:
                 raise ValueError(f"Number of columns in X ({X_array.shape[1]}) must match the number of features the model was trained on ({total_trained_features}).")
        else:
            # If model not fitted, it's safer to raise an error or handle accordingly
            raise RuntimeError("Model has not been fitted yet. Call fit() before predict().")

        return self.model.predict(X_array, categorical=self.categorical_indices)

    def _validate_data(self, X: Union[pd.DataFrame, np.ndarray]) -> np.ndarray:
        """
        Ensures input data is a NumPy array.
        """
        if isinstance(X, pd.DataFrame):
            return X.values
        elif isinstance(X, np.ndarray):
            return X
        else:
            raise ValueError("Input data X must be a pandas DataFrame or a NumPy array.")

    def evaluate(self, X_test: Union[pd.DataFrame, np.ndarray] = None,
                     y_test: Union[pd.Series, np.ndarray] = None) -> Dict[str, Any]:
        """
        Evaluates the clustering performance.

        Args:
            X_test: Optional test data for external evaluation.
            y_test: Optional ground truth labels for external evaluation.

        Returns:
            Dictionary of evaluation metrics.
        """
        results: Dict[str, Any] = {}
        if not hasattr(self.model, 'cost_'):
            print("Warning: Model not fitted or cost_ attribute missing.")
            return results

        results['cost'] = self.model.cost_
        results['numerical_centroids'] = self.model.cluster_centroids_[0].tolist()
        results['categorical_centroids'] = self.model.cluster_centroids_[1].tolist()

        if X_test is not None and y_test is not None:
            try:
                cluster_labels = self.predict(X_test)
                results['adjusted_rand_score'] = adjusted_rand_score(y_test, cluster_labels)
                results['normalized_mutual_info_score'] = normalized_mutual_info_score(y_test, cluster_labels)
            except ValueError as e:
                print(f"Warning: Could not calculate external evaluation metrics. Ensure X_test has the correct number of features. Error: {e}")
            except RuntimeError as e: # Catch the RuntimeError from predict if model not fitted
                print(f"Warning: Model not fitted, cannot predict for evaluation. Error: {e}")

        return results

    def get_centroids(self) -> Union[Tuple[List[List[float]], List[List[Any]]], None]:
        """
        Returns the cluster centroids.

        Returns:
            Tuple of numerical and categorical centroids, or None if not fitted.
        """
        if hasattr(self.model, 'cluster_centroids_'):
            return (self.model.cluster_centroids_[0].tolist(), self.model.cluster_centroids_[1].tolist())
        else:
            return None

    def plot_clusters(self, X: Union[pd.DataFrame, np.ndarray], title: str = "K-Prototypes Clustering") -> None:
        """
        Visualizes the clusters. This method provides a basic visualization
        by plotting numerical features and using hue for clusters. For categorical
        features, it prints value counts per cluster.

        Args:
            X: The data used for clustering (features).
            title: Title of the plot.
        """
        if not hasattr(self.model, 'labels_'):
            raise RuntimeError("Model has not been fitted yet. Call fit() before plotting.")

        X_df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=self.feature_names)
        cluster_labels = self.model.labels_
        X_df['cluster'] = cluster_labels

        numerical_features = [col for i, col in enumerate(X_df.columns) if i not in self.categorical_indices and col != 'cluster']
        categorical_features = [col for i, col in enumerate(X_df.columns) if i in self.categorical_indices]

        # --- Numerical Features Visualization ---
        if numerical_features:
            print(f"\n--- Visualizing Numerical Features for {title} ---")
            num_numerical_plots = len(numerical_features)
            # Determine grid size for subplots
            cols = 3 # Max 3 columns per row
            rows = (num_numerical_plots + cols - 1) // cols
            fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
            axes = axes.flatten() # Flatten for easy iteration

            for i, feature in enumerate(numerical_features):
                sns.boxplot(x='cluster', y=feature, data=X_df, ax=axes[i], palette='viridis')
                axes[i].set_title(f'{feature} by Cluster')
                axes[i].set_xlabel('Cluster')
                axes[i].set_ylabel(feature)

            # Hide unused subplots
            for j in range(i + 1, len(axes)):
                fig.delaxes(axes[j])

            plt.tight_layout()
            plt.suptitle(f"Distribution of Numerical Features Across Clusters\n{title}", y=1.02, fontsize=16)
            plt.show()

            # Pair plot for first few numerical features if many
            if len(numerical_features) >= 2:
                print("\n--- Pair Plot of First 2 Numerical Features (if applicable) ---")
                try:
                    sns.pairplot(X_df, vars=numerical_features[:2], hue='cluster', palette='viridis', diag_kind='kde')
                    plt.suptitle(f"Pair Plot of Numerical Features by Cluster\n{title}", y=1.02, fontsize=16)
                    plt.show()
                except Exception as e:
                    print(f"Could not generate pair plot (might need more than one numerical feature or data issues): {e}")


        # --- Categorical Features Visualization ---
        if categorical_features:
            print(f"\n--- Visualizing Categorical Features for {title} ---")
            num_categorical_plots = len(categorical_features)
            # Determine grid size for subplots
            cols = 2 # Max 2 columns per row for categorical
            rows = (num_categorical_plots + cols - 1) // cols
            fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
            axes = axes.flatten()

            for i, feature in enumerate(categorical_features):
                # Count plot to show distribution of categories within each cluster
                sns.countplot(x=feature, hue='cluster', data=X_df, ax=axes[i], palette='viridis')
                axes[i].set_title(f'{feature} Distribution by Cluster')
                axes[i].set_xlabel(feature)
                axes[i].set_ylabel('Count')
                axes[i].tick_params(axis='x', rotation=45) # Rotate labels if needed

            # Hide unused subplots
            for j in range(i + 1, len(axes)):
                fig.delaxes(axes[j])

            plt.tight_layout()
            plt.suptitle(f"Distribution of Categorical Features Across Clusters\n{title}", y=1.02, fontsize=16)
            plt.show()

        # --- Centroid Visualization (more abstract) ---
        print("\n--- Cluster Centroids ---")
        numerical_centroids, categorical_centroids = self.get_centroids()
        if numerical_centroids is not None and categorical_centroids is not None:
            print("\nNumerical Centroids:")
            num_cols = [self.feature_names[i] for i in range(len(self.feature_names)) if i not in self.categorical_indices]
            print(pd.DataFrame(numerical_centroids, columns=num_cols, index=[f'Cluster {i}' for i in range(self.n_clusters)]))

            print("\nCategorical Centroids:")
            cat_cols = [self.feature_names[i] for i in self.categorical_indices]
            # Convert categorical centroids to a more readable format (e.g., actual categories if possible)
            # For simplicity, here we just print the raw categorical centroids from kmodes
            # which are internal representations.
            print(pd.DataFrame(categorical_centroids, columns=cat_cols, index=[f'Cluster {i}' for i in range(self.n_clusters)]))
        else:
            print("Centroids are not available. Model might not be fitted.")

        print("\n--- Cluster Sizes ---")
        print(X_df['cluster'].value_counts().sort_index())

In [51]:
import joblib
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
import numpy as np
import pandas as pd
from typing import Dict, Union, Any

class LinearModels(BaseModel):
    """
    A wrapper class for scikit-learn's Linear Models
    for both regression and classification tasks.
    NOTE: This version requires manual preprocessing of data (X) before fitting or predicting.
    It does NOT handle numerical scaling or categorical encoding internally.
    """

    def __init__(self,
                 problem_type: str,
                 model_name: str,
                 grid_search: bool = True,
                 random_state: int = None,
                 custom_param_grid: Dict = None,
                 **kwargs):
        self.problem_type = problem_type.lower()
        self.model_name = model_name.lower()
        self.grid_search = grid_search
        self.random_state = random_state
        self.custom_param_grid = custom_param_grid

        self.model: Union[Any, GridSearchCV, None] = None
        self.best_model: Union[Any, None] = None
        self.cv_results: Union[Dict, None] = None

        self.model = self._create_model(**kwargs)


    def _create_model(self, **kwargs) -> Union[GridSearchCV, Any]:
        model_params = {}
        grid_params = {}

        essential_model_config_params = {
            'fit_intercept', 'positive', 'copy_X', 'n_jobs',
            'max_iter', 'tol', 'warm_start', 'precompute', 'selection',
            'solver', 'class_weight'
        }

        grid_search_config_params = {
            'cv', 'scoring', 'n_jobs', 'verbose', 'error_score', 'return_train_score'
        }

        if self.random_state is not None:
            if self.model_name in ['logistic', 'ridge', 'lasso', 'elasticnet']:
                if 'random_state' not in kwargs:
                    model_params['random_state'] = self.random_state

        for key, value in kwargs.items():
            if key in essential_model_config_params:
                model_params[key] = value
            elif key in grid_search_config_params:
                grid_params[key] = value
            else:
                model_params[key] = value

        base_model = None
        if self.problem_type == 'classification':
            if self.model_name == 'logistic':
                model_params.setdefault('class_weight', 'balanced')
                model_params.setdefault('max_iter', 1000)
                if self.grid_search:
                    model_params.setdefault('solver', 'saga')
                else:
                    model_params.setdefault('solver', 'lbfgs')
                base_model = LogisticRegression(**model_params)
            else:
                raise ValueError(f"Model '{self.model_name}' not supported for classification.")
        elif self.problem_type == 'regression':
            if self.model_name == 'linear':
                base_model = LinearRegression(**model_params)
            elif self.model_name == 'ridge':
                base_model = Ridge(**model_params)
            elif self.model_name == 'lasso':
                base_model = Lasso(**model_params)
            elif self.model_name == 'elasticnet':
                base_model = ElasticNet(**model_params)
            else:
                raise ValueError(f"Model '{self.model_name}' not supported for regression.")
        else:
            raise ValueError("problem_type must be 'classification' or 'regression'")

        if self.grid_search:
            cv = grid_params.get('cv', 5)
            scoring = grid_params.get('scoring', 'accuracy' if self.problem_type == 'classification' else 'neg_mean_squared_error')
            n_jobs = grid_params.get('n_jobs', -1)
            verbose = grid_params.get('verbose', 0)
            error_score = grid_params.get('error_score', np.nan)
            return_train_score = grid_params.get('return_train_score', True)

            param_grid = self.custom_param_grid if self.custom_param_grid is not None else self._get_param_grid()

            grid_search_model = GridSearchCV(base_model, param_grid, cv=cv, scoring=scoring, n_jobs=n_jobs, verbose=verbose, error_score=error_score, return_train_score=return_train_score)
            return grid_search_model
        else:
            return base_model

    def _get_param_grid(self):
        if self.problem_type == 'classification' and self.model_name == 'logistic':
            return [
                # Penalidade L1
                {
                    'C': [0.01, 0.1, 1, 10, 100],
                    'penalty': ['l1'],
                    'solver': ['saga'],
                    # Removido l1_ratio: [1.0] para evitar o UserWarning, pois 'l1' implica l1_ratio=1.0
                },
                # Penalidade L2
                {
                    'C': [0.01, 0.1, 1, 10, 100],
                    'penalty': ['l2'],
                    'solver': ['saga', 'lbfgs', 'newton-cg', 'sag'], # 'saga' pode ser lento para grandes datasets aqui
                                                                       # lbfgs e newton-cg são boas escolhas para l2
                },
                # Penalidade ElasticNet
                {
                    'C': [0.01, 0.1, 1, 10, 100],
                    'penalty': ['elasticnet'],
                    'solver': ['saga'],
                    'l1_ratio': [0.1, 0.5, 0.9]
                },
                # Sem Penalidade
                {
                    'C': [0.01, 0.1, 1, 10, 100],
                    'penalty': [None],
                    'solver': ['lbfgs', 'newton-cg', 'sag', 'saga'] # 'saga' ainda é válido para None, mas outras opções são eficientes
                }
            ]
        elif self.problem_type == 'regression':
            # ... (seu código atual para regressão) ...
            pass
        return {}

    def fit(self, X: pd.DataFrame, y: pd.Series):
        self.best_model = None
        self.cv_results = None

        if self.grid_search:
            print(f"Iniciando GridSearchCV para {self.model_name.upper()}...")
            try:
                self.model.fit(X, y)
                self.best_model = self.model.best_estimator_
                self.cv_results = self.model.cv_results_
                print(f"GridSearchCV finalizado para {self.model_name.upper()}. Melhores parâmetros: {self.model.best_params_}")
            except Exception as e:
                print(f"ERRO: GridSearchCV para {self.model_name.upper()} falhou durante o fit: {e}")
        else:
            print(f"Treinando modelo {self.model_name.upper()}...")
            try:
                self.model.fit(X, y)
                self.best_model = self.model
                print(f"Modelo {self.model_name.upper()} treinado.")
            except Exception as e:
                print(f"ERRO: Treinamento direto do modelo {self.model_name.upper()} falhou: {e}")

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        if self.best_model is None:
            raise RuntimeError("Modelo não foi treinado ainda. Chame .fit() primeiro.")
        return self.best_model.predict(X)

    def evaluate(self, X_test: pd.DataFrame, y_test: pd.Series) -> Dict:
        if self.best_model is None:
            print("AVISO: Modelo não treinado ou falhou no treinamento. Não é possível avaliar.")
            return {}

        y_pred = self.predict(X_test)
        results = {}
        if self.problem_type == 'regression':
            results['mean_squared_error'] = mean_squared_error(y_test, y_pred)
            results['r2_score'] = r2_score(y_test, y_pred)
            results['rmse'] = np.sqrt(results['mean_squared_error'])
        elif self.problem_type == 'classification':
            results['accuracy'] = accuracy_score(y_test, y_pred)
            if len(np.unique(y_test)) > 1:
                results['classification_report'] = classification_report(y_test, y_pred, zero_division=0)
                results['confusion_matrix'] = confusion_matrix(y_test, y_pred)
            else:
                print("AVISO: Apenas uma classe presente em y_test. classification_report e confusion_matrix não calculados.")
                results['classification_report'] = "N/A"
                results['confusion_matrix'] = "N/A"
        return results

    def get_feature_importance(self) -> Union[np.ndarray, None]:
        if self.best_model and hasattr(self.best_model, 'coef_'):
            return self.best_model.coef_
        else:
            print("Importância das features (coeficientes) não disponível para este modelo ou kernel.")
            return None

In [52]:
import numpy as np
from sklearn import metrics
from typing import Optional, List
import pandas as pd

class ModelEvaluator:
    """
    A class for evaluating machine learning models.  It supports binary, multiclass,
    and regression tasks, and provides a variety of evaluation metrics,
    cross-validation, and visualization options.

    Attributes:
        model: The trained machine learning model to evaluate.  Can be any object
            that implements a `predict` method for classification or regression,
            and optionally a `predict_proba` method for classification.
        task_type:  The type of machine learning task.  Must be one of
            'binary', 'multiclass', or 'regression'.
        metrics: A list of metric names (strings) to calculate.
            See the `supported_metrics` property for available metrics.
            Defaults to ['accuracy'] for classification and ['r2'] for regression.
        random_state:  The random state to use for any random operations.
            Defaults to 42.
        pos_label:  The positive class label for binary classification.  Used for
            metrics like precision, recall, and F1-score.  Defaults to 1.
    """
    supported_metrics = {
        'binary': [
            'accuracy', 'precision', 'recall', 'f1', 'auc', 'average_precision',
            'log_loss', 'brier_score'
        ],
        'multiclass': [
            'accuracy', 'precision_macro', 'precision_micro', 'recall_macro',
            'recall_micro', 'f1_macro', 'f1_micro', 'log_loss'
        ],
        'regression': [
            'r2', 'mse', 'rmse', 'mae', 'explained_variance', 'max_error'
        ]
    }

    def __init__(
        self,
        model: object,
        task_type: str,
        metrics: Optional[List[str]] = None,
        random_state: int = 42,
        pos_label: int = 1,
    ):
        self.model = model
        self.task_type = task_type
        self.metrics = metrics if metrics else (
            ['accuracy'] if task_type in ['binary', 'multiclass'] else ['r2']
        )
        self.random_state = random_state
        self.pos_label = pos_label

        self._validate_task_type()
        self._validate_metrics()

    def _validate_task_type(self):
        """Validates that the task type is one of the supported types."""
        if self.task_type not in ['binary', 'multiclass', 'regression']:
            raise ValueError(
                f"Invalid task type: {self.task_type}.  Must be one of "
                "'binary', 'multiclass', or 'regression'."
            )

    def _validate_metrics(self):
        """Validates that the specified metrics are supported for the task type."""
        for metric in self.metrics:
            if metric not in self.supported_metrics[self.task_type]:
                raise ValueError(
                    f"Invalid metric: {metric} for task type: {self.task_type}.  "
                    f"Supported metrics are: {self.supported_metrics[self.task_type]}"
                )

    def evaluate(
        self,
        X: pd.DataFrame,
        y: pd.Series,
        return_predictions: bool = False
    ) -> dict:
        """
        Evaluates the model on the given data.

        Args:
            X: The input features as a pandas DataFrame.
            y: The target values as a pandas Series.
            return_predictions: Whether to return the predictions and (if applicable)
                predicted probabilities along with the metrics. Defaults to False.

        Returns:
            A dictionary containing the calculated metrics on the provided dataset.
            If return_predictions is True, the dictionary also includes
            'predictions' and (for classification) 'probabilities' keys.
        """
        return self._evaluate_without_cv(X, y, return_predictions)

    def _evaluate_without_cv(
        self,
        X: pd.DataFrame,
        y: pd.Series,
        return_predictions: bool
    ) -> dict:
        """
        Evaluates the model on the given data.

        Args:
            X: The input features.
            y: The target values.
            return_predictions: Whether to return predictions.

        Returns:
            A dictionary containing the calculated metrics.
            Optionally includes the predictions.
        """
        # Assuming the model is already trained if no CV is used by the evaluator itself.
        # If the model needs training, it should be done before calling evaluate.
        y_pred = self.model.predict(X)

        if self.task_type in ['binary', 'multiclass']:
            try:
                y_prob = self.model.predict_proba(X)
            except AttributeError:
                y_prob = None
                if 'log_loss' in self.metrics:
                    print(
                        "Warning: Model does not have predict_proba method. "
                        "Log loss cannot be calculated."
                    )
        else:
            y_prob = None

        results = self._calculate_metrics(y, y_pred, y_prob)
        if return_predictions:
            results['predictions'] = y_pred
            if self.task_type in ['binary', 'multiclass']:
                results['probabilities'] = y_prob
        return results

    def _calculate_metrics(
        self,
        y_true: pd.Series,
        y_pred: np.ndarray,
        y_prob: Optional[np.ndarray] = None
    ) -> dict:
        """
        Calculates the specified evaluation metrics.

        Args:
            y_true: The true target values.
            y_pred: The predicted target values.
            y_prob: The predicted probabilities (optional, required for some metrics).

        Returns:
            A dictionary containing the calculated metrics.
        """
        metrics_dict = {}
        for metric in self.metrics:
            if metric == 'accuracy':
                metrics_dict[metric] = metrics.accuracy_score(y_true, y_pred)
            elif metric in ['precision', 'recall', 'f1']:
                average = 'binary' if self.task_type == 'binary' else 'macro'
                if metric == 'precision':
                    metrics_dict[metric] = metrics.precision_score(
                        y_true, y_pred, average=average, pos_label=self.pos_label, zero_division=0
                    )
                elif metric == 'recall':
                    metrics_dict[metric] = metrics.recall_score(
                        y_true, y_pred, average=average, pos_label=self.pos_label, zero_division=0
                    )
                elif metric == 'f1':
                    metrics_dict[metric] = metrics.f1_score(
                        y_true, y_pred, average=average, pos_label=self.pos_label, zero_division=0
                    )
            elif metric == 'auc':
                if self.task_type == 'binary':
                    metrics_dict[metric] = metrics.roc_auc_score(y_true, y_prob[:, 1])
                else:
                    # Handle multiclass case.
                    metrics_dict[metric] = metrics.roc_auc_score(y_true, y_prob, multi_class='ovr')
            elif metric == 'average_precision':
                metrics_dict[metric] = metrics.average_precision_score(
                    y_true, y_prob[:, 1]
                )
            elif metric == 'log_loss' and y_prob is not None:
                metrics_dict[metric] = metrics.log_loss(y_true, y_prob)
            elif metric == 'brier_score':
                metrics_dict[metric] = metrics.brier_score_loss(y_true, y_prob[:, 1])
            elif metric in ['precision_macro', 'precision_micro', 'recall_macro',
                             'recall_micro', 'f1_macro', 'f1_micro']:
                average = metric.split('_')[-1]  # Extract 'macro' or 'micro'
                if 'precision' in metric:
                    metrics_dict[metric] = metrics.precision_score(y_true, y_pred, average=average, zero_division=0)
                elif 'recall' in metric:
                    metrics_dict[metric] = metrics.recall_score(y_true, y_pred, average=average, zero_division=0)
                elif 'f1' in metric:
                    metrics_dict[metric] = metrics.f1_score(y_true, y_pred, average=average, zero_division=0)
            elif metric == 'r2':
                metrics_dict[metric] = metrics.r2_score(y_true, y_pred)
            elif metric == 'mse':
                metrics_dict[metric] = metrics.mean_squared_error(y_true, y_pred)
            elif metric == 'rmse':
                metrics_dict[metric] = np.sqrt(metrics.mean_squared_error(y_true, y_pred))
            elif metric == 'mae':
                metrics_dict[metric] = metrics.mean_absolute_error(y_true, y_pred)
            elif metric == 'explained_variance':
                metrics_dict[metric] = metrics.explained_variance_score(y_true, y_pred)
            elif metric == 'max_error':
                metrics_dict[metric] = metrics.max_error(y_true, y_pred)
        return metrics_dict


In [53]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report, r2_score
from sklearn.utils import compute_class_weight  # This might not be used if class_weights are passed directly as tensor
from torch.utils.data import DataLoader, TensorDataset


class NeuralNetworkTrainer:
    def __init__(self, model, task, lr=0.001, class_weights=None):
        self.model = model
        self.task = task
        self.optimizer = optim.Adam(model.parameters(), lr=lr)

        # Initialize criterion based on task type
        if task == 'classification':
            # Ensure class_weights is a tensor if used with CrossEntropyLoss
            if class_weights is not None:
                # If class_weights is a numpy array, convert it to a torch tensor
                if isinstance(class_weights, torch.Tensor):
                    self.criterion = nn.CrossEntropyLoss(weight=class_weights)
                else:  # Assuming class_weights is a list or numpy array
                    self.criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32))
            else:
                self.criterion = nn.CrossEntropyLoss()
        elif task == 'regression':
            self.criterion = nn.MSELoss()
        else:
            raise ValueError("Task must be 'regression' or 'classification'")

    def train(self, X_train, y_train, epochs=100, batch_size=32, sampler=None):
        self.model.train()  # Set the model to training mode
        dataset = TensorDataset(X_train, y_train)

        # IMPORTANT FIX: Add drop_last=True to DataLoader
        if sampler:
            # If using a custom sampler, ensure it can handle dropping last batch if needed
            # For typical use-cases, drop_last=True is generally safe.
            loader = DataLoader(dataset, batch_size=batch_size, sampler=sampler, drop_last=True)
        else:
            # This is the most common path; ensure drop_last=True here
            loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)  # KEY CHANGE HERE

        print(f"Starting training for {epochs} epochs with batch size {batch_size}...")
        for epoch in range(epochs):
            for X_batch, y_batch in loader:
                self.optimizer.zero_grad()  # Zero the gradients before each batch
                outputs = self.model(X_batch)  # Forward pass

                # Reshape y_batch for regression if it's not already [batch_size, 1]
                if self.task == 'regression' and outputs.shape != y_batch.shape:
                    y_batch = y_batch.view_as(outputs)

                loss = self.criterion(outputs, y_batch)  # Calculate loss
                loss.backward()  # Backward pass
                self.optimizer.step()  # Update model weights

            # Print loss every 10 epochs (or adjust frequency)
            if epoch % 10 == 0:
                print(f"Epoch {epoch} - Loss: {loss.item():.4f}")
        print("Training finished.")

    def predict(self, X):
        self.model.eval()  # Set model to evaluation mode
        with torch.no_grad():  # Disable gradient calculations
            outputs = self.model(X)
            if self.task == 'classification':
                # For classification, return the predicted class index
                return torch.argmax(outputs, dim=1)
            else:
                # For regression, ensure the output matches the expected shape (e.g., flatten if needed)
                return outputs.squeeze()  # Removes dimensions of size 1 (e.g., [N, 1] -> [N])

    def evaluate(self, X_test, y_test, batch_size=32):
        self.model.eval()  # Set model to evaluation mode
        dataset = TensorDataset(X_test, y_test)
        # For evaluation, drop_last is not typically necessary unless you have specific batch requirements
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

        all_preds = []
        all_labels = []

        with torch.no_grad():
            for X_batch, y_batch in loader:
                outputs = self.model(X_batch)
                if self.task == 'classification':
                    preds = torch.argmax(outputs, dim=1)
                else:
                    preds = outputs.squeeze()  # Match predict output format
                all_preds.append(preds)
                all_labels.append(y_batch)

        y_pred = torch.cat(all_preds)
        y_true = torch.cat(all_labels)

        if self.task == 'classification':
            print("\n=== Classification Report ===")
            # Ensure y_true and y_pred are numpy arrays for sklearn metrics
            print(classification_report(y_true.cpu().numpy(), y_pred.cpu().numpy(), zero_division=0))
        elif self.task == 'regression':
            # Ensure y_true and y_pred are numpy arrays for sklearn metrics
            mse = mean_squared_error(y_true.cpu().numpy(), y_pred.cpu().numpy())
            r2 = r2_score(y_true.cpu().numpy(), y_pred.cpu().numpy())
            print("\n=== Regression Evaluation ===")
            print(f"Mean Squared Error: {mse:.4f}")
            print(f"R² Score: {r2:.4f}")

In [54]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix


class RandomForest(BaseModel):
    """
    A wrapper class for scikit-learn's RandomForest algorithms
    for both regression and classification tasks, implementing Bagging of Decision Trees.
    Inherits from BaseModel.
    """
    def __init__(self, problem_type='classification', **kwargs):
        """
        Initializes the RandomForest class.

        Args:
            problem_type (str, optional): The type of problem ('classification' or 'regression').
                                         Defaults to 'classification'.
            **kwargs: Additional arguments to be passed to the scikit-learn
                      RandomForest Classifier or Regressor. Common arguments include:
                      n_estimators (number of trees), criterion, max_depth,
                      min_samples_split, min_samples_leaf, max_features,
                      bootstrap (True for bagging), oob_score, n_jobs, random_state, etc.
        """
        self.problem_type = problem_type.lower()
        self.model = self._create_model(**kwargs)

    def _create_model(self, **kwargs):
        """
        Creates the appropriate RandomForest model instance
        based on the problem type.
        """
        if self.problem_type == 'classification':
            return RandomForestClassifier(**kwargs)
        elif self.problem_type == 'regression':
            return RandomForestRegressor(**kwargs)
        else:
            raise ValueError("problem_type must be 'classification' or 'regression'")

    def fit(self, X, y):
        """
        Trains the RandomForest model with the provided training data.

        Args:
            X (array-like): Training data (features).
            y (array-like): Training labels or target values.
        """
        self.model.fit(X, y)

    def predict(self, X):
        """
        Performs predictions on the given data.

        Args:
            X (array-like): Data for prediction (features).

        Returns:
            array-like: Model predictions.
        """
        return self.model.predict(X)

    def predict_proba(self, X):
        """
        Predicts class probabilities for classification tasks.

        Args:
            X (array-like): Data for prediction (features).

        Returns:
            array-like: Predicted class probabilities.

        Raises:
            NotImplementedError: If the problem type is not 'classification'.
        """
        if self.problem_type != 'classification':
            raise NotImplementedError("predict_proba is only available for classification tasks.")
        return self.model.predict_proba(X)

    def evaluate(self, X_test, y_test):
        """
        Evaluates the performance of the model on the provided test data.

        Args:
            X_test (array-like): Test data (features).
            y_test (array-like): Test labels or target values.

        Returns:
            dict: A dictionary containing the appropriate evaluation metrics
                  for the problem type.
        """
        y_pred = self.predict(X_test)
        results = {}
        if self.problem_type == 'regression':
            results['mean_squared_error'] = mean_squared_error(y_test, y_pred)
            results['r2_score'] = r2_score(y_test, y_pred)
            # You could add more regression metrics here if needed, e.g., MAE, RMSE
            # from sklearn.metrics import mean_absolute_error
            # results['mean_absolute_error'] = mean_absolute_error(y_test, y_pred)
            # results['root_mean_squared_error'] = mean_squared_error(y_test, y_pred, squared=False) # For RMSE
        elif self.problem_type == 'classification':
            results['accuracy'] = accuracy_score(y_test, y_pred)
            results['classification_report'] = classification_report(y_test, y_pred, output_dict=True) # output_dict=True makes it easier to parse
            results['confusion_matrix'] = confusion_matrix(y_test, y_pred).tolist() # Convert to list for easier handling if serializing
            # You could add more classification metrics here if needed, e.g., precision, recall, f1-score for specific classes or averaged
            # from sklearn.metrics import precision_score, recall_score, f1_score
            # results['precision_macro'] = precision_score(y_test, y_pred, average='macro')
            # results['recall_macro'] = recall_score(y_test, y_pred, average='macro')
            # results['f1_macro'] = f1_score(y_test, y_pred, average='macro')
        return results

    def get_feature_importance(self):
        """
        Returns the feature importances.

        Returns:
            numpy.ndarray: An array containing the importance of each feature.
                           Returns None if the model does not support feature importance (though RandomForest does).
        """
        # RandomForest models have feature_importances_ attribute
        if hasattr(self.model, 'feature_importances_'):
            return self.model.feature_importances_
        else:
            return None


# Carregamento do dataset e avaliação das mértricas dos modelos

In [55]:
import os
from data_analysis.analizer import DataAnalizer

# ==========================================
#  Setup do diretório de output dos modelos
# ==========================================
model_output_dir = "../out/models/"
os.makedirs(model_output_dir, exist_ok=True)


# ==================
#  Setup do dataset
# ==================

df = pd.read_csv("../out/dataset.csv")

# a grande maioria das entradas têm valores nulos em "congestion_surcharge"
del df["congestion_surcharge"]

# dados continuos (não é possivel normalizar estes dados)
del df["pickup_time_in_seconds"]
del df["dropoff_time_in_seconds"]

# Remove all rows with any missing values (NaN)
df= df.dropna()

# =============================================
# 1. REGRESSION DATASET (continuous target)
# =============================================

# Separate features and target
X_reg = df.drop(columns=['fare_amount'])
y_reg = df['fare_amount']

# Scale only the features (not target)
scaler = StandardScaler()
X_reg_scaled = scaler.fit_transform(X_reg)

# Create scaled DataFrame for regression
df_regression = pd.DataFrame(X_reg_scaled, columns=X_reg.columns)
df_regression['fare_amount'] = y_reg.values  # Add unscaled target

analizer_reg = DataAnalizer(df_regression, "fare_amount", test_size=0.02)


# =============================================
# 2. CLASSIFICATION DATASET (categorical target)
# =============================================

# Create fare classes
bins = [-np.inf, 10, 30, 60, np.inf]
labels = [1, 2, 3, 4]

# Create classification target
df_classification = df.copy()
df_classification['fare_class'] = pd.cut(
    df['fare_amount'],
    bins=bins,
    labels=labels
)

# Separate features and target
X_clf = df_classification.drop(columns=['fare_amount', 'fare_class'])
y_clf = df_classification['fare_class']

# Scale features using SAME scaler (important for consistency)
X_clf_scaled = scaler.transform(X_clf)  # Use existing scaler

# Create scaled DataFrame for classification
df_classification_scaled = pd.DataFrame(X_clf_scaled, columns=X_clf.columns)
df_classification_scaled['fare_class'] = y_clf.values  # Add target

analizer_clf = DataAnalizer(df_classification_scaled, "fare_class", test_size=0.2)

# =============================================
# Verification
# =============================================
print("Regression dataset:")
print(df_regression.head())

print("\nClassification dataset:")
print(df_classification_scaled.head())

print(df_classification['fare_class'].value_counts(normalize=True))

Data divided successfully.
Data divided successfully.
Regression dataset:
   trip_distance  tip_amount  tolls_amount     extra  passenger_count  \
0      -0.666634   -0.760919     -0.229742 -0.463238        -0.470065   
1      -0.404075    0.619532     -0.229742 -0.463238        -0.470065   
2      -0.455557   -0.174228     -0.229742 -0.463238        -0.470065   
3      -0.149237   -0.760919     -0.229742 -0.463238        -0.470065   
4       1.665513    0.964644     -0.229742 -0.463238        -0.470065   

   pickup_hour  pickup_day_of_week  pickup_day_of_month  pickup_month  \
0    -2.316139           -1.018927            -1.674574     -1.532716   
1    -2.316139           -1.018927            -1.674574     -1.532716   
2    -2.316139           -1.018927            -1.674574     -1.532716   
3    -2.316139           -1.018927            -1.674574     -1.532716   
4    -2.316139           -1.018927            -1.674574     -1.532716   

   dropoff_hour  dropoff_day_of_week  dropoff_da

In [85]:
# =========================
# Loading Supervised Models
# =========================

classification_models = {
    "KNN": KNN.deserialize(model_output_dir + "KNN-Classifier-best.pkl"),
    "DecisionTree": DecisionTree.deserialize(model_output_dir + "decisionTreeClassifier.pkl"),
    "Linear Regression": LinearModels.deserialize(model_output_dir + "logisticClassifier.pkl"),
    "Random Forest": RandomForest.deserialize(model_output_dir + "randomForestClassifier.pkl"),
    "AdaBoost": AdaBoost.deserialize(model_output_dir + "AdaBoost-Classifier.pkl")
}

regression_models = {
    "KNN": KNN.deserialize(model_output_dir + "KNN-Regressor-best.pkl"),
    "DecisionTree": DecisionTree.deserialize(model_output_dir + "decisionTreeRegressor.pkl"),
    "Logistic Regression": LinearModels.deserialize(model_output_dir + "linearRegressor_best.pkl"),
    "Random Forest": RandomForest.deserialize(model_output_dir + "randomForestRegressor.pkl"),
    "AdaBoost": AdaBoost.deserialize(model_output_dir + "AdaBoost-Regressor.pkl")
}

clustering_models = {
    "K-Prototypes": KPrototypesModel.deserialize(model_output_dir + "KPrototypesClustering_nClusters.pkl"),
    "HDBSCAN": HDBSCANMixed.deserialize(model_output_dir + "HDCSCAN_Clustering.pkl"),
}

In [61]:
# =======================================
# Analyzing metrics for regression models
# =======================================

for model_name in regression_models:
    try:
        print("Evaluating model:", model_name)
        evaluator = ModelEvaluator(
            model=regression_models[model_name],
            task_type="regression",
            metrics=["rmse", "mae", "r2"]
        ).evaluate(X=analizer_reg.data_test, y=analizer_reg.labels_test)

        print("\nKNN Regression Model Evaluation Results:")
        for metric, value in evaluator.items():
            if isinstance(value, (int, float, np.number)):
                print(f"  {metric}: {value:.4f}")

    except Exception as e:
        print(f"\nError loading or predicting: {e}")

    print("\n===============================\n")

Evaluating model: KNN

KNN Regression Model Evaluation Results:
  rmse: 4.1336
  mae: 2.1840
  r2: 0.8894


Evaluating model: DecisionTree

KNN Regression Model Evaluation Results:
  rmse: 2.2622
  mae: 1.3244
  r2: 0.9669


Evaluating model: Logistic Regression

KNN Regression Model Evaluation Results:
  rmse: 6.1558
  mae: 2.3152
  r2: 0.7547


Evaluating model: Random Forest

KNN Regression Model Evaluation Results:
  rmse: 1.3944
  mae: 0.6255
  r2: 0.9874


Evaluating model: AdaBoost

KNN Regression Model Evaluation Results:
  rmse: 5.6424
  mae: 2.7063
  r2: 0.7939


Evaluating model: Neural Network

Error loading or predicting: linear(): argument 'input' (position 1) must be Tensor, not DataFrame




In [58]:
# ===========================================
# Analyzing metrics for classification models
# ===========================================

for model_name in classification_models:
    try:
        print("Evaluating model:", model_name)

        evaluator = ModelEvaluator(
            model=classification_models[model_name],
            task_type="multiclass",
            metrics=["accuracy", "precision_macro", "recall_macro", "f1_macro"]
        ).evaluate(X=analizer_clf.data_test, y=analizer_clf.labels_test)

        print(f"\n{model_name} Classification Model Evaluation Results:")
        for metric, value in evaluator.items():
            if isinstance(value, (int, float, np.number)):
                print(f"  {metric}: {value:.4f}")

    except Exception as e:
        print(f"\nError evaluating {model_name} classification model: {e}")

    print("\n===============================\n")

Evaluating model: KNN

KNN Classification Model Evaluation Results:
  accuracy: 0.7883
  precision_macro: 0.8160
  recall_macro: 0.7257
  f1_macro: 0.7541


Evaluating model: DecisionTree

DecisionTree Classification Model Evaluation Results:
  accuracy: 0.9662
  precision_macro: 0.9608
  recall_macro: 0.9653
  f1_macro: 0.9630


Evaluating model: Linear Regression

Linear Regression Classification Model Evaluation Results:
  accuracy: 0.8610
  precision_macro: 0.6960
  recall_macro: 0.8391
  f1_macro: 0.7378


Evaluating model: Random Forest

Random Forest Classification Model Evaluation Results:
  accuracy: 0.9763
  precision_macro: 0.9775
  recall_macro: 0.9659
  f1_macro: 0.9716


Evaluating model: AdaBoost

AdaBoost Classification Model Evaluation Results:
  accuracy: 0.8762
  precision_macro: 0.8321
  recall_macro: 0.7961
  f1_macro: 0.8128


Evaluating model: Neural Network

Error evaluating Neural Network classification model: 'collections.OrderedDict' object has no attribute '

In [73]:
# =======================
# Loading Neural Networks
# =======================

nn_clf_input_dim = analizer_clf.data_test.shape[1]
nn_clf_output_classes = 4

# 1. Instantiate your BaseNeuralNetwork with the correct dimensions for the classifier.
nn_classifier_model_instance = BaseNeuralNetwork(
    input_dim=nn_clf_input_dim,
    output_dim=nn_clf_output_classes,
    task_type='classification'
)

# 2. Load the saved weights (state_dict) into this newly created model instance.
#    'map_location' ensures it loads correctly regardless of where it was saved (CPU/GPU).
nn_classifier_state_dict = torch.load(model_output_dir + "NeuralNetworkClassifier.pth", map_location=torch.device('cpu'))
nn_classifier_model_instance.load_state_dict(nn_classifier_state_dict)

# 3. Set the model to evaluation mode (important for consistent predictions with BatchNorm/Dropout).
nn_classifier_model_instance.eval()

# 4. Create an instance of your NeuralNetworkTrainer, passing in the loaded model.
#    Your ModelEvaluator will call the '.predict()' method of this trainer object.
nn_classifier_ready_for_eval = NeuralNetworkTrainer(model=nn_classifier_model_instance, task='classification')


# --- Prepare the Neural Network Regressor ---
# Determine input_dim for the regressor based on your test data features.
# The output_dim for regression is typically 1 (a single continuous value).
nn_reg_input_dim = analizer_reg.data_test.shape[1]

# 1. Instantiate your BaseNeuralNetwork with the correct dimensions for the regressor.
nn_regressor_model_instance = BaseNeuralNetwork(
    input_dim=nn_reg_input_dim,
    output_dim=1, # Output dimension is 1 for regression
    task_type='regression'
)

# 2. Load the saved weights (state_dict) into this newly created model instance.
nn_regressor_state_dict = torch.load(model_output_dir + "NeuralNetworkRegressor.pth", map_location=torch.device('cpu'))
nn_regressor_model_instance.load_state_dict(nn_regressor_state_dict)

# 3. Set the model to evaluation mode.
nn_regressor_model_instance.eval()

# 4. Create an instance of your NeuralNetworkTrainer, passing in the loaded model.
nn_regressor_ready_for_eval = NeuralNetworkTrainer(model=nn_regressor_model_instance, task='regression')

In [81]:
# =============================
# Classification Neural Network
# =============================

import torch
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, log_loss,
    mean_squared_error, r2_score, mean_absolute_error, explained_variance_score, max_error,
    classification_report
)

print("--- Manually Evaluating Neural Network Classifier ---")

X_test_clf_tensor = torch.tensor(analizer_clf.data_test.values, dtype=torch.float32)

y_pred_clf = nn_classifier_ready_for_eval.predict(X_test_clf_tensor)

nn_classifier_model_instance.eval()
with torch.no_grad():
    raw_outputs_clf = nn_classifier_model_instance(X_test_clf_tensor)
    y_prob_clf = raw_outputs_clf.cpu().numpy()

# Ensure true labels are in NumPy array format for sklearn AND 0-indexed
# This aligns with the conversion used in your training code (astype(int) - 1)
y_true_clf = analizer_clf.labels_test.values.astype(int) - 1

print(f"Accuracy: {accuracy_score(y_true_clf, y_pred_clf):.4f}")
print(f"F1-Macro: {f1_score(y_true_clf, y_pred_clf, average='macro', zero_division=0):.4f}")
print(f"Precision-Macro: {precision_score(y_true_clf, y_pred_clf, average='macro', zero_division=0):.4f}")
print(f"Recall-Macro: {recall_score(y_true_clf, y_pred_clf, average='macro', zero_division=0):.4f}")
print(f"Log Loss: {log_loss(y_true_clf, y_prob_clf):.4f}")

print("\nFull Classification Report:")
print(classification_report(y_true_clf, y_pred_clf, zero_division=0))


print("\n--- Manually Evaluating Neural Network Regressor ---")

X_test_reg_tensor = torch.tensor(analizer_reg.data_test.values, dtype=torch.float32)

y_pred_reg = nn_regressor_ready_for_eval.predict(X_test_reg_tensor)

# Ensure true labels are in NumPy array format for sklearn (no special conversion needed here based on your training)
y_true_reg = analizer_reg.labels_test.values

print(f"R² Score: {r2_score(y_true_reg, y_pred_reg):.4f}")
print(f"Mean Squared Error (MSE): {mean_squared_error(y_true_reg, y_pred_reg):.4f}")
print(f"Root Mean Squared Error (RMSE): {np.sqrt(mean_squared_error(y_true_reg, y_pred_reg)):.4f}")
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_true_reg, y_pred_reg):.4f}")
print(f"Explained Variance Score: {explained_variance_score(y_true_reg, y_pred_reg):.4f}")
print(f"Max Error: {max_error(y_true_reg, y_pred_reg):.4f}")

--- Manually Evaluating Neural Network Classifier ---
Accuracy: 0.8373
F1-Macro: 0.7348
Precision-Macro: 0.6754
Recall-Macro: 0.8912
Log Loss: 2.0463

Full Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.83      0.88      9490
           1       0.77      0.84      0.80      6138
           2       0.67      0.92      0.78      1142
           3       0.32      0.98      0.48       106

    accuracy                           0.84     16876
   macro avg       0.68      0.89      0.73     16876
weighted avg       0.86      0.84      0.84     16876


--- Manually Evaluating Neural Network Regressor ---
R² Score: -1207.5476
Mean Squared Error (MSE): 186671.1196
Root Mean Squared Error (RMSE): 432.0545
Mean Absolute Error (MAE): 23.6911
Explained Variance Score: -1204.2393
Max Error: 14688.5969
